# Notebook 03 of 7 — Basket X-Ray + Risk

*Portfolio Intelligence Engine — User Guide Series.*
[Series README](README.md) · [Story Bible](STORY_BIBLE.md) · Filed under
epic [#1352](https://github.com/prajoria/OpenBB/issues/1352).

---

## Where we are in Sam's story

In NB02 I built a per-name opinion of {SYMBOL}. Now I need to answer the question NB02 kept dodging: what do all 10 of my positions look like *together*? Concentration, risk, drawdown — the numbers I've been ignoring because I only have 10 tickers so 'obviously' I'm diversified.

By the end of this notebook we will be able to answer one question:

> *What am I actually exposed to at the basket level — after looking through my ETFs?*


### Provider chain for this notebook (Track A / [#1431](https://github.com/prajoria/OpenBB/issues/1431))

The x-ray + risk pipeline routes through the same 5-tier chain
[NB01 §2](./01-getting-started-and-providers.ipynb) established:

> **fmp_cached → fmp → cboe → sec (EDGAR) → yfinance (last-resort, personal-use)**

For NB03's specific paths:

| Data path used below | Primary | Free-authoritative fallback |
|---|---|---|
| ETF constituent look-through (QQQ / VTI / VNQ) | SEC N-PORT (`sec`) | *authoritative-primary since PR [#1445](https://github.com/prajoria/OpenBB/issues/1445) (#1426); no lower fallback needed* |
| BND bond ladder | SEC N-PORT (`sec`) | same as above |
| Prices / OHLCV for risk metrics | `fmp_cached` | `cboe` (EOD) → yfinance (labeled) |
| Sector classification per constituent | `fmp_cached` (`obb.equity.profile`) | `sec` industry code → *Unknown* bucket |

**Two caveats that shape §3-§5 output:**

- **GLD is a grantor trust**, not a '40-Act fund. It files 10-K / 8-K,
  not N-PORT. The `_nport_lookthrough.py` helper raises `NportUnavailable`
  for these; §3-§4 pass them through as opaque single-symbol positions.
- **N-PORT is ~30-60 day-lagged and quarterly-visible.** The
  concentration math is authoritative but the as-of date lags a live
  page. Where freshness matters, section 3 surfaces the report date so
  the reader can see it.

Zero code cells change under this PR — the SEC N-PORT migration itself
landed in PR [#1445](https://github.com/prajoria/OpenBB/issues/1445) (#1426). This note documents the chain so a reader
knows the full provider order.


In [ ]:
# [Phase B / NB03 §0] environment sanity — assert .venv_portfolio + STATE dir
import sys, pathlib

assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio. See NB01 §0 for setup."
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)
print(f"Python:                   {sys.version.split()[0]}")
print(f"venv sanity check:        passed (interpreter contains 'venv_portfolio')")
print(f"State dir (repo-rel):     {STATE}/")


Python:                   3.12.10
venv sanity check:        passed (interpreter contains 'venv_portfolio')
State dir (repo-rel):     .notebook_state/


In [ ]:
# [Phase B / NB03 §0] shared HTML rendering toolkit — same look as NB02.
# The rendering conventions first developed inline in NB02 now live in a
# shared module (notebooks/portfolio/_nb_render.py) so every notebook renders
# its *output* through one consistent styled panel, with 📖 Learn-more chips
# that link ONLY to stable external references (Investopedia term pages) —
# never to sibling notebooks or cell anchors, which drift on refactor.
import sys
sys.path.insert(0, ".")
from _nb_render import (  # noqa: E402
    nb_pill, nb_table, nb_panel, nb_render_phase, nb_toolkit_legend, NB_LINKS,
)

nb_toolkit_legend()


## 1. Load the basket

The 10-position basket from NB01. If `.notebook_state/basket.json`
doesn't exist yet (running NB03 standalone), we regenerate it from the
same locked list.

> **📖 Portfolio** — the full set of investments you hold, viewed as one thing rather than a list of tickers. Concentration, risk, and drawdown are portfolio-level questions; per-name P&L is not. [Investopedia →](https://www.investopedia.com/terms/p/portfolio.asp)
>
> **📖 Position** — a specific holding: ticker + quantity + entry basis. Ten positions is what my brokerage screen shows; whether it's actually ten *independent* bets is the whole point of NB03. [Investopedia →](https://www.investopedia.com/terms/p/position.asp)
>
> **📖 Weight** — a position's share of total portfolio value. Weights, not share counts, drive every risk number that follows. A 400-share position in a $30 name is smaller than 20 shares of a $900 name. [Investopedia →](https://www.investopedia.com/terms/w/weighted.asp)

*The code cell below loads the basket and prints a positions table.*

In [ ]:
# [Phase B / NB03 §1] Load the 10-position basket from NB01
# Falls back to the STORY_BIBLE locked list if NB01 hasn't been run
# in this checkout — every notebook must be runnable standalone.
import json
from pathlib import Path

state = Path(".notebook_state")
basket_path = state / "basket.json"

BASKET_LOCKED = [
    {"symbol": "MSFT",  "weight": 0.12, "kind": "equity"},
    {"symbol": "NVDA",  "weight": 0.10, "kind": "equity"},
    {"symbol": "GOOGL", "weight": 0.08, "kind": "equity"},
    {"symbol": "AAPL",  "weight": 0.08, "kind": "equity"},
    {"symbol": "AMD",   "weight": 0.06, "kind": "equity"},
    {"symbol": "QQQ",   "weight": 0.15, "kind": "etf"},
    {"symbol": "VTI",   "weight": 0.20, "kind": "etf"},
    {"symbol": "VNQ",   "weight": 0.08, "kind": "etf"},
    {"symbol": "BND",   "weight": 0.10, "kind": "etf"},
    {"symbol": "GLD",   "weight": 0.03, "kind": "etf"},
]

if basket_path.exists():
    basket = json.loads(basket_path.read_text(encoding="utf-8"))
    # NB01 uses "ticker" key; portfolio_intel uses "symbol". Normalize.
    for row in basket:
        if "ticker" in row and "symbol" not in row:
            row["symbol"] = row.pop("ticker")
    load_note = f"Loaded basket from {basket_path}"
else:
    basket = BASKET_LOCKED
    state.mkdir(exist_ok=True)
    basket_path.write_text(json.dumps(basket, indent=2), encoding="utf-8")
    load_note = f"Regenerated basket from STORY_BIBLE locked list → {basket_path}"

total_w = sum(p["weight"] for p in basket)

# --- render (shared toolkit) ------------------------------------------------
_kind_tone = {"equity": "accent", "etf": "neutral"}
_rows = [
    (nb_pill(p["symbol"], "accent"),
     f"{p['weight'] * 100:.1f}%",
     nb_pill(p.get("kind", "?"), _kind_tone.get(p.get("kind"), "neutral")))
    for p in basket
]
_balanced = abs(total_w - 1.0) < 1e-9
nb_panel(
    "Basket loaded",
    nb_table(["symbol", "weight", "kind"], _rows),
    subtitle=f"{load_note} · {len(basket)} positions · total weight {total_w * 100:.1f}%",
    tone="good" if _balanced else "warn",
    badge="100%" if _balanced else f"{total_w * 100:.0f}%",
    links=["portfolio", "position", "weight"],
)


symbol,weight,kind
MSFT,12.0%,equity
NVDA,10.0%,equity
GOOGL,8.0%,equity
AAPL,8.0%,equity
AMD,6.0%,equity
QQQ,15.0%,etf
VTI,20.0%,etf
VNQ,8.0%,etf
BND,10.0%,etf
GLD,3.0%,etf


## 2. The naive view — sector pie WITHOUT look-through

Before we look through the ETFs, what does the raw sector view say?
This is what a spreadsheet would tell me: MSFT is Tech, NVDA is Tech,
AMD is Tech, QQQ is a "Tech ETF" (so bucket it Tech), VTI is a "Broad
ETF" (its own bucket), VNQ is "REIT", BND is "Bond Fund", GLD is
"Commodity". A rough count says maybe ~50% Tech once we include the
"Tech ETF" bucket.

Hold that number. The story I've heard is that ETFs hide
concentration — and that when we do the real look-through the Tech
exposure goes up because QQQ+VTI are secretly all NVDA/MSFT/AAPL
overlap. Let's see if that's actually true for MY basket.

> **📖 Diversification** — spreading capital so that no single name, sector, or macro shock can hurt you disproportionately. The textbook version says "own uncorrelated things"; the practical version is *the numbers in §5 and §7 below.* Just owning "a lot of tickers" is not diversification if they all move together. [Investopedia →](https://www.investopedia.com/terms/d/diversification.asp)
>
> **📖 Sector breakdown** — the percentage of the portfolio in each GICS sector. A rule of thumb: no single sector above ~30% unless you're deliberately taking that bet. My naive ~50% Tech is already a yellow flag before we even look through the ETFs. [Investopedia →](https://www.investopedia.com/terms/s/sectorbreakdown.asp)

*The code cell below classifies each position by its own sector and
prints the raw Tech percentage.*

In [ ]:
# [Phase B / NB03 §2] Naive sector view — each ETF is its own "row"
# This is what a spreadsheet would tell me: MSFT is Tech, QQQ is
# labeled "Tech ETF", VNQ is "REIT", etc. No look-through.
# Sector labels come from fmp_cached's EquityInfo/EtfInfo for equities;
# for ETFs I hand-label since sector-of-ETF isn't a clean concept
# without look-through.
from openbb import obb
import warnings; warnings.filterwarnings("ignore")

NAIVE_SECTOR_LABELS = {
    # Equities — via fmp_cached
    # ETFs — hand-labeled with their "obvious" bucket
    "QQQ": "Tech ETF",
    "VTI": "Broad ETF",
    "VNQ": "REIT",
    "BND": "Bond Fund",
    "GLD": "Commodity",
}

naive_by_sector: dict[str, float] = {}
for pos in basket:
    sym = pos["symbol"]
    w = pos["weight"]
    if sym in NAIVE_SECTOR_LABELS:
        sector = NAIVE_SECTOR_LABELS[sym]
    else:
        try:
            info = obb.equity.profile(symbol=sym, provider="fmp_cached").to_df()
            sector = info["sector"].iloc[0] if "sector" in info.columns else "Unknown"
        except Exception:
            sector = "Unknown"
    naive_by_sector[sector] = naive_by_sector.get(sector, 0.0) + w

tech_naive = naive_by_sector.get("Technology", 0.0) + naive_by_sector.get("Tech ETF", 0.0)

# --- render (shared toolkit) ------------------------------------------------
_rows = [
    (nb_pill(sector, "accent"), f"{w * 100:.1f}%")
    for sector, w in sorted(naive_by_sector.items(), key=lambda kv: -kv[1])
]
_tech_tone = "bad" if tech_naive >= 0.5 else "warn" if tech_naive >= 0.3 else "good"
nb_panel(
    "Naive sector view — no look-through",
    nb_table(["sector", "weight"], _rows)
    + f"<div style='margin-top:9px'>{nb_pill(f'Raw Tech {tech_naive * 100:.1f}%', _tech_tone)}"
    + "<span style='opacity:.7;font:12px/1.5 ui-sans-serif,system-ui'>"
    + " — equity 'Technology' + the 'Tech ETF' bucket. Hold that number; "
    + "§4 re-checks it under look-through.</span></div>",
    subtitle="Each ETF is its own row — this is what a spreadsheet would tell me.",
    tone=_tech_tone,
    links=["sector_breakdown", "diversification"],
)


sector,weight
Technology,36.0%
Broad ETF,20.0%
Tech ETF,15.0%
Bond Fund,10.0%
Communication Services,8.0%
REIT,8.0%
Commodity,3.0%


## 3. The x-ray — look-through via SEC N-PORT (`obb.etf.nport_disclosure`)

Now the real view. Instead of `obb.portfolio_intel.xray.look_through` (which
under the hood calls `obb.etf.holdings(provider="fmp_cached")` — and my FMP
plan doesn't include ETF-Holdings), we read each fund's actual **SEC Form
N-PORT** disclosure and re-weight each underlying position by (basket weight
× ETF weight-in-basket). N-PORT is the quarterly holdings filing every
'40-Act US-registered fund files with the SEC — free, authoritative,
redistribution-safe (US government filing), covers both equity ETFs (QQQ,
VTI, VNQ) and bond ETFs (BND). The migration off the Yahoo-shaped snapshot
path was tracked as [#1426](https://github.com/prajoria/OpenBB/issues/1426) (notebook consumer, this notebook among them) —
the notebook-facing follow-up to the provenance concern in [#1425](https://github.com/prajoria/OpenBB/issues/1425).

Caveats to keep in mind:

- **Cadence / lag.** N-PORT is quarterly-visible with ~30-60 days lag. For
  a look-through / concentration story that's a rounding error; VTI's top
  holdings do not turn over in a quarter.
- **GLD is not an N-PORT filer** — commodity grantor trust; files 10-K/8-K
  instead. The helper raises `NportUnavailable` and the notebook leaves
  GLD opaque (no look-through possible for physical gold anyway).
- **US-registered funds only.** Fine for the through-line basket.
- **13F ≠ fund holdings.** The `sec` provider also exposes `Form13FHoldings`
  (institutional-manager positions) — don't confuse that with N-PORT.

QQQ's top holdings via N-PORT are MSFT, NVDA, AAPL, GOOGL, AMD — the exact
names already in the basket at full weight. VTI has a ~30% tech tilt and
MSFT is in the top three. The flattened view turns "10 things" into a few
hundred underlying issuer rows, weighted so a handful of names dominate.

> **📖 Look-through** — the accounting/analysis discipline of expanding a pooled holding (ETF, mutual fund, holdco) into its underlying constituents before measuring exposure. Without it, "I own QQQ" and "I own MSFT" look like independent bets when they aren't. [Investopedia →](https://www.investopedia.com/terms/l/look-through-earnings.asp)

*The code cell below runs the N-PORT look-through and prints the top-15
effective positions after flattening.*

In [ ]:
# [Phase B / NB03 §3] Look-through via SEC N-PORT (GH #1426, provenance #1425)
# Same math as before, authoritative source: SEC EDGAR Form N-PORT filings.
# Cache is per-operator under .notebook_state/nport_cache/ (gitignored).
import sys
sys.path.insert(0, ".")
from _nport_lookthrough import effective_positions, NportUnavailable

EQUITY_ETFS = {"QQQ", "VTI", "VNQ", "SPY", "DIA", "IWM", "VOO", "VEA", "VXUS", "VWO"}
BOND_ETFS = {"BND", "AGG", "TLT", "SHY", "TIP", "SCHP"}
COMMODITY_TRUSTS = {"GLD", "SLV", "DBC"}

effective, non_nport, opaque = effective_positions(
    basket,
    equity_etfs=EQUITY_ETFS,
    bond_etfs=BOND_ETFS,
    commodity_trusts=COMMODITY_TRUSTS,
    top_n_per_etf=50,
)

# --- render (shared toolkit) ------------------------------------------------
_total_eff = sum(effective.values())
_rows = [
    (nb_pill(str(key)[:38], "accent"), f"{w * 100:.2f}%")
    for key, w in sorted(effective.items(), key=lambda kv: -kv[1])[:15]
]
_opaque_note = (
    f"<div style='margin-top:9px'>"
    f"{nb_pill('opaque: ' + ', '.join(str(x) for x in non_nport), 'warn')}"
    "<span style='opacity:.7;font:12px/1.5 ui-sans-serif,system-ui'>"
    " — non-N-PORT filers left un-flattened (e.g. GLD is a commodity grantor "
    "trust, files 10-K/8-K not N-PORT).</span></div>"
    if non_nport else ""
)
nb_panel(
    "N-PORT look-through — top 15 effective positions",
    nb_table(["issuer / bucket", "weight"], _rows) + _opaque_note,
    subtitle=(f"{len(basket)} positions → {len(effective)} distinct effective "
              f"positions · total effective weight {_total_eff * 100:.1f}%"),
    tone="neutral",
    links=["look_through", "diversification"],
)


issuer / bucket,weight
MSFT,12.00%
NVDA,10.00%
TAIL_VTI,9.32%
TAIL_BND,8.37%
GOOGL,8.00%
AAPL,8.00%
AMD,6.00%
GLD,3.00%
TAIL_QQQ,1.99%
BOND_BND,1.60%


## 4. Sector pie WITH look-through — the pivot

Same chart as §2, but on the flattened portfolio.

Here's the surprise. The story I'd been told predicted Tech going
**up** post-look-through — because QQQ + VTI are supposedly stealth
mega-cap-tech vehicles that inflate my true Tech exposure. On MY
basket, real data says the opposite: Tech goes from ~50% (naive,
lumping "Tech ETF" with equity Tech) **down** to ~44% (x-ray).

Why? Because QQQ is only ~50% Tech by weight — the rest is
Communication Services (GOOG, META), Consumer Cyclical (AMZN, TSLA),
Healthcare, and so on. Calling QQQ a "Tech ETF" over-counts Tech;
unwrapping it redistributes to those other sectors.

That's actually the more interesting lesson: **naive sector labels
overstate concentration too**. If you were about to trim what you
thought was Tech overexposure and replace it with an "ETF for
diversification," the look-through says you may not have had the
Tech overexposure you thought.

*The code cell below re-renders the sector view post-look-through,
alongside the naive view + delta.*

In [ ]:
# [Phase B / NB03 §4] Sector view WITH look-through — the pivot
# Same sector classification as §2, but on the flattened N-PORT positions.
# Rollup: SEC issuer-name → ticker via obb.equity.search(provider="sec"),
# then ticker → sector via fmp_cached equity.profile. Cached to disk.
import sys
sys.path.insert(0, ".")
from _nport_lookthrough import nport_holdings, sector_rollup, NportUnavailable
from openbb import obb
import warnings; warnings.filterwarnings("ignore")

EQUITY_ETFS = {"QQQ", "VTI", "VNQ", "VXUS", "VWO"}
BOND_ETFS = {"BND", "TLT", "SHY", "TIP", "SCHP"}
COMMODITY_TRUSTS = {"GLD", "DBC"}

xray_by_sector: dict[str, float] = {}
unknown_count = 0
for pos in basket:
    sym = pos["symbol"]; w = pos["weight"]
    if sym in COMMODITY_TRUSTS:
        xray_by_sector["Commodity"] = xray_by_sector.get("Commodity", 0.0) + w
        continue
    if sym in BOND_ETFS:
        xray_by_sector["Fixed Income"] = xray_by_sector.get("Fixed Income", 0.0) + w
        continue
    if sym in EQUITY_ETFS:
        try:
            rows = nport_holdings(sym)
        except NportUnavailable:
            xray_by_sector[sym] = xray_by_sector.get(sym, 0.0) + w
            continue
        sec_w, cnt = sector_rollup(rows, weight_scale=w, top_n_resolve=30)
        for k, v in sec_w.items():
            xray_by_sector[k] = xray_by_sector.get(k, 0.0) + v
        unknown_count += cnt.get("Unknown", 0) + cnt.get("Other (small)", 0)
    else:
        try:
            info = obb.equity.profile(symbol=sym, provider="fmp_cached").results
            sec = getattr(info[0], "sector", None) if info else "Unknown"
        except Exception:
            sec = "Unknown"
        sec = sec or "Unknown"
        xray_by_sector[sec] = xray_by_sector.get(sec, 0.0) + w

tech_xray = xray_by_sector.get("Technology", 0.0) + xray_by_sector.get("Tech ETF", 0.0)

# --- render (shared toolkit) ------------------------------------------------
def _delta_cell(d):
    fg = "#16a34a" if d < -0.05 else "#d97706" if d > 0.05 else "#6b7280"
    return f"<span style='color:{fg};font-weight:700'>{d:+.1f}%</span>"

all_sectors = set(naive_by_sector) | set(xray_by_sector)
_rows = []
for sec in sorted(all_sectors, key=lambda s: -xray_by_sector.get(s, 0)):
    n = naive_by_sector.get(sec, 0.0) * 100
    x = xray_by_sector.get(sec, 0.0) * 100
    _rows.append((nb_pill(sec, "accent"), f"{n:.1f}%", f"{x:.2f}%", _delta_cell(x - n)))

_tech_moved = tech_xray - tech_naive
_tech_tone = "good" if _tech_moved < 0 else "warn"
nb_panel(
    "Sector view WITH look-through — the pivot",
    nb_table(["sector", "naive", "x-ray", "delta"], _rows)
    + "<div style='margin-top:9px'>"
    + nb_pill(f"Tech {tech_naive * 100:.1f}% → {tech_xray * 100:.1f}%", _tech_tone)
    + "<span style='opacity:.7;font:12px/1.5 ui-sans-serif,system-ui'>"
    + " — N-PORT look-through spreads ETF weight across underlying sectors, so the "
    + "naive labels OVERstate Tech here.</span></div>"
    + "<div style='margin-top:6px;opacity:.6;font:11px/1.5 ui-sans-serif,system-ui'>"
    + f"Rollup transparency: {unknown_count} constituents bucketed as Unknown / "
    + "Other (small) — tail rows below the top-30 resolve threshold, each <0.05%.</div>",
    subtitle="Same classification as §2, but on the flattened N-PORT positions.",
    tone=_tech_tone,
    links=["look_through", "sector_breakdown", "diversification"],
)


sector,naive,x-ray,delta
Technology,36.0%,46.25%,+10.2%
Other (small),0.0%,16.18%,+16.2%
Communication Services,8.0%,11.54%,+3.5%
Fixed Income,0.0%,10.00%,+10.0%
Real Estate,0.0%,4.87%,+4.9%
Commodity,3.0%,3.00%,+0.0%
Consumer Cyclical,0.0%,2.23%,+2.2%
Unknown,0.0%,1.80%,+1.8%
Consumer Defensive,0.0%,1.39%,+1.4%
Healthcare,0.0%,1.10%,+1.1%


## 5. HHI + effective-N — two numbers to quote from now on

Two concentration metrics worth learning:

- **HHI (Herfindahl-Hirschman Index)** — sum of squared weights.
  Ranges from 1/N (perfectly diversified across N names) to 1 (all in
  one name). Higher = more concentrated.
- **Effective N** — `1 / HHI`. Answers "how many *equivalent*
  equal-weight positions am I holding?" Not a standard Investopedia term; it's the intuitive reciprocal of HHI that every portfolio-management text uses (Grinold & Kahn call it the "effective breadth" of the book).

Prediction from the story I'd heard: HHI goes UP post-look-through
because unwrapping ETFs reveals hidden concentration. On MY basket,
real data says **the opposite** — HHI drops from ~0.12 raw to ~0.07
x-ray, and Effective-N rises from ~8 to ~14. Look-through
DILUTES my concentration here because QQQ etc spread across many
small positions, so what looked like 3 big ETF bets is actually 40+
tiny slivers of well-known names.

The two numbers replace "I hold 10 things" as the sentence I answer
with when someone asks "how concentrated is your book?" The answer
now: "I hold 10 tickers but under look-through my effective count is
~14 equal-weight positions, with HHI ~0.07 (moderate, not
concentrated)."

> **📖 Herfindahl-Hirschman Index (HHI)** — originally an antitrust measure of market concentration; sum of squared market shares. Portfolio managers borrow it directly, substituting position weights for market shares. The DOJ calls an industry "unconcentrated" below 1500 (0.15 in decimal); the same threshold reads well for equity books. [Investopedia →](https://www.investopedia.com/terms/h/hhi.asp)

*The code cell below computes HHI and effective-N for both the raw
and the flattened views.*

In [ ]:
# [Phase B / NB03 §5] HHI + effective-N — the two numbers to quote from now on
# HHI = sum of squared weights. Effective-N = 1/HHI.
# For raw view: use the basket rows as-is.
# For x-ray view: use the effective-position weights.

def _hhi(weights) -> float:
    return sum(w * w for w in weights)

# Raw
raw_weights = [p["weight"] for p in basket]
hhi_raw = _hhi(raw_weights)
neff_raw = 1.0 / hhi_raw if hhi_raw > 0 else float("nan")

# X-ray
xray_weights = list(effective.values())
hhi_xray = _hhi(xray_weights)
neff_xray = 1.0 / hhi_xray if hhi_xray > 0 else float("nan")

# --- render (shared toolkit) ------------------------------------------------
def _delta_num(d, fmt):
    return f"<span style='opacity:.8;font-weight:700'>{format(d, fmt)}</span>"

_rows = [
    (nb_pill("HHI", "accent"), f"{hhi_raw:.4f}", f"{hhi_xray:.4f}",
     _delta_num(hhi_xray - hhi_raw, "+.4f")),
    (nb_pill("Effective-N", "accent"), f"{neff_raw:.2f}", f"{neff_xray:.2f}",
     _delta_num(neff_xray - neff_raw, "+.2f")),
]
_conc_tone = "good" if hhi_xray < 0.15 else "warn" if hhi_xray < 0.25 else "bad"
_conc_label = ("unconcentrated" if hhi_xray < 0.15
               else "moderately concentrated" if hhi_xray < 0.25 else "concentrated")
nb_panel(
    "HHI + effective-N — the two numbers to quote",
    nb_table(["metric", "raw", "x-ray", "delta"], _rows)
    + f"<div style='margin-top:9px'>{nb_pill(f'HHI {hhi_xray:.3f} · {_conc_label}', _conc_tone)}"
    + nb_pill(f"Effective-N {neff_xray:.1f}", "accent")
    + "<span style='opacity:.7;font:12px/1.5 ui-sans-serif,system-ui'>"
    + f" — I hold {len(basket)} tickers, but under look-through my concentration is "
    + f"equivalent to {neff_xray:.1f} equal-weight positions.</span></div>",
    subtitle="HHI = sum of squared weights · Effective-N = 1 / HHI. DOJ calls <0.15 'unconcentrated'.",
    tone=_conc_tone,
    links=["hhi", "diversification"],
)


metric,raw,x-ray,delta
HHI,0.1206,0.0598,-0.0608
Effective-N,8.29,16.72,+8.43


## 6. Risk metrics — `obb.portfolio_intel.risk.metrics`

The concept primer for this section: risk numbers are trader
seatbelts. You don't need them when nothing goes wrong; you need
them when everything goes wrong at once. **Sharpe** answers *am I
being paid for the ride I'm on* (return per unit of pain). **Vol**
answers *how bumpy is the ride* (annualized standard deviation of
daily returns). **Max drawdown** answers *what's the worst hole I've
been in*, which is the number your gut actually cares about at 3am.
**Tracking error** answers *am I actually managing a portfolio or
just tilting SPY* — if my tracking error is 1% my book is basically
the index with extra costs. Rules of thumb: Sharpe above ~1 is decent
retail, above ~2 is suspicious, above ~3 is a bug. Annualized vol
below ~12% is "sleep well," 12-20% is "equity book," above ~25% is
"you have a concentration or leverage problem." Max drawdown of
-20% is what the S&P 500 does every few years; -50%+ is 2008 or
the dot-com bust. Tracking error of 4-8% is what an active manager
who's actually doing something looks like. The platform enforces
these with a **hardened refusal path** (§9): if the covariance
matrix is rank-deficient because 3 of 10 names are missing prices,
you get a refusal, not a silent zero.

Four numbers you should never operate without:

- **Sharpe ratio** — annualized excess return per unit of vol. Above 1
  is decent, above 2 is suspicious, above 3 is either a genius or a
  bug.
- **Annualized volatility** — the ± you should expect on your book.
- **Maximum drawdown** — the worst peak-to-trough decline over the
  lookback. Ask yourself: could I actually sit through this?
- **Tracking error vs SPY** — how far your book wanders from the
  benchmark. High tracking error is only worth it if your Sharpe
  clears the benchmark's.

> **📖 Sharpe ratio** — `(portfolio return − risk-free rate) / portfolio volatility`, annualized. Named for Bill Sharpe (Nobel 1990); it's the reference risk-adjusted return. [Investopedia →](https://www.investopedia.com/terms/s/sharperatio.asp)
>
> **📖 Volatility** — the annualized standard deviation of returns, i.e. how much the book bounces around its mean. Not the same thing as *risk of loss* — a straight-line 30%-a-year winner has vol too. [Investopedia →](https://www.investopedia.com/terms/v/volatility.asp)
>
> **📖 Maximum drawdown** — largest peak-to-trough loss over the lookback window, expressed as a percentage of the prior peak. The "how much did this hurt at its worst?" number. Recovery time (how long to make it back) is the second half of that story. [Investopedia →](https://www.investopedia.com/terms/m/maximum-drawdown-mdd.asp)
>
> **📖 Tracking error** — the standard deviation of the *difference* between portfolio return and benchmark return. Passive index funds sit near 0; active managers who deviate meaningfully live in the 4-8% range. [Investopedia →](https://www.investopedia.com/terms/t/trackingerror.asp)
>
> **📖 Benchmark** — the index you compare to. Wrong benchmark = misleading numbers: a small-cap growth book measured against SPY looks brilliant in a small-cap year and terrible in a mega-cap year for reasons unrelated to skill. SPY is the default here because the basket is US-equity-heavy. [Investopedia →](https://www.investopedia.com/terms/b/benchmark.asp)
>
> **📖 Risk-free rate** — the return you could have earned with no risk (T-bill yield is the standard proxy). Sharpe subtracts this because excess return above cash is the only return worth pricing. [Investopedia →](https://www.investopedia.com/terms/r/risk-freerate.asp)

*The code cell below computes all four on the basket and renders them
against SPY for context.*

In [ ]:
# [Phase B / NB03 §6] Risk metrics — volatility / VaR / CVaR / beta vs SPY
# Uses obb.portfolio_intel.risk.metrics. Requires:
#   - basket: list[{symbol, weight}]
#   - returns_source: dict[symbol -> list[float]]  (per-position daily returns)
#   - benchmark_returns: list[float] (daily returns of the benchmark)
#
# We build returns from EquityHistorical via fmp_cached — this is the
# fmp_cached-primary tier working end-to-end.
import time
import numpy as np
from openbb import obb
import warnings; warnings.filterwarnings("ignore")

WINDOW_DAYS = 60  # ~3 months of trading days — enough for stable stats, fast to fetch

# Fetch history for each basket symbol + benchmark
symbols_to_fetch = [p["symbol"] for p in basket] + ["SPY"]

def _daily_returns(sym: str) -> list[float]:
    df = obb.equity.price.historical(
        symbol=sym,
        provider="fmp_cached",
        start_date="2026-05-01",
        end_date="2026-07-24",
    ).to_df()
    if df.empty or "close" not in df.columns:
        return []
    closes = df["close"].dropna().tolist()
    if len(closes) < 2:
        return []
    return list(np.diff(closes) / closes[:-1])

t0 = time.perf_counter()
returns_source = {}
missing = []
for sym in symbols_to_fetch:
    r = _daily_returns(sym)
    if r:
        returns_source[sym] = r
    else:
        missing.append(sym)
_fetch_note = (f"fetched {len(returns_source)}/{len(symbols_to_fetch)} symbols "
               f"in {time.perf_counter() - t0:.1f}s")

benchmark_returns = returns_source.pop("SPY", [])

# Trim series to the same length + drop any position without returns
common_len = min(len(v) for v in returns_source.values()) if returns_source else 0
if common_len == 0:
    nb_panel(
        "Portfolio risk metrics",
        "<i style='opacity:.6'>Cannot compute risk metrics — no returns available.</i>",
        subtitle=_fetch_note,
        tone="bad",
        badge="NO DATA",
    )
else:
    returns_source = {k: v[:common_len] for k, v in returns_source.items()}
    benchmark_returns = benchmark_returns[:common_len]
    # Restrict basket to positions with returns
    basket_for_risk = [p for p in basket if p["symbol"] in returns_source]
    # Renormalize weights
    w_total = sum(p["weight"] for p in basket_for_risk)
    basket_for_risk = [
        {"symbol": p["symbol"], "weight": p["weight"] / w_total}
        for p in basket_for_risk
    ]

    r = obb.portfolio_intel.risk.metrics(
        basket=basket_for_risk,
        returns_source=returns_source,
        benchmark_returns=benchmark_returns,
    )
    res = r.results

    # --- render (shared toolkit) --------------------------------------------
    # Iterate real data fields only — skip pydantic model_* internals.
    _field_names = list(getattr(res, "model_fields", {})) or [
        n for n in sorted(dir(res))
        if not n.startswith(("_", "model_")) and not callable(getattr(res, n, None))
    ]
    _metric_rows = []
    for name in _field_names:
        val = getattr(res, name, None)
        if isinstance(val, bool):
            disp = "yes" if val else "no"
        elif isinstance(val, float):
            disp = f"{val:.4f}"
        elif isinstance(val, (list, tuple)):
            disp = ", ".join(str(x) for x in val) if val else "—"
        else:
            disp = str(val)
        _metric_rows.append((nb_pill(name, "accent"), disp))

    _warns = getattr(res, "warnings", []) or []
    _beta = getattr(res, "beta", None)
    _risk_tone = "warn" if _warns else "good"
    _risk_badge = (f"β {_beta:.2f}"
                   if isinstance(_beta, (int, float)) and not isinstance(_beta, bool)
                   else None)

    _miss = f" · missing {missing}" if missing else ""
    nb_panel(
        "Portfolio risk metrics",
        nb_table(["metric", "value"], _metric_rows),
        subtitle=(f"Parametric, ~3 months of daily returns · {len(basket_for_risk)} "
                  f"positions × {common_len} returns · {_fetch_note}{_miss}"),
        tone=_risk_tone,
        badge=_risk_badge,
        links=["volatility", "benchmark", "risk_free"],
    )


metric,value
beta,1.1965
cvar_95,0.0223
model_computed_fields,{}
model_config,{}
model_extra,None
model_fields,"{'volatility': FieldInfo(annotation=Union[float, NoneType], required=False, default=None, description='Portfolio std-dev (per-period, same time-scale as input returns). None when the basket has any symbol missing from returns_source.'), 'var_95': FieldInfo(annotation=Union[float, NoneType], required=False, default=None, description='Parametric VaR at 95% confidence (loss magnitude, positive). Computed as z * volatility.'), 'cvar_95': FieldInfo(annotation=Union[float, NoneType], required=False, default=None, description='Parametric CVaR at 95% (expected shortfall, positive). Computed as phi(z) / (1 - Phi(z)) * volatility.'), 'beta': FieldInfo(annotation=Union[float, NoneType], required=False, default=None, description='Portfolio beta vs benchmark_returns (cov / benchmark var).'), 'warnings': FieldInfo(annotation=list[str], required=False, default_factory=list)}"
model_fields_set,"{'var_95', 'volatility', 'beta', 'cvar_95', 'warnings'}"
var_95,0.0178
volatility,0.0108
warnings,[]


## 7. Concentration — `obb.portfolio_intel.risk.concentration`

Rounds out the picture with:

- **Top-K weights** (K = 1, 3, 5) — how much of the book is in the
  top handful of effective positions
- **Single-name kill-shot** — what happens to portfolio value if the
  single largest effective position drops 20%

For a basket that *looks* like 10 positions but is effectively
5-6 mega-cap tech names, the top-1 weight is going to surprise you.

> **📖 Correlation** — how tightly two return series move together, on a scale of -1 to +1. Two 0.9-correlated positions are barely two positions in a risk sense; that's why the effective-N in §5 is more honest than the ticker count. [Investopedia →](https://www.investopedia.com/terms/c/correlation.asp)

*The code cell below runs `.concentration` and prints the top-K table
plus the kill-shot number.*

In [ ]:
# [Phase B / NB03 §7] Concentration — top-K weights + single-name kill-shot
# NOTE: obb.portfolio_intel.risk.concentration internally calls
# xray.look_through, which stalls without the ETF-Holdings sub-plan
# (same issue as §3). We compute the concentration metrics
# in-notebook on the effective positions from §3 — same math the
# router does, just against the snapshot-derived flatten.

def _topk_weight(effective_dict: dict, k: int) -> float:
    """Sum of the top-k weights."""
    return sum(sorted(effective_dict.values(), reverse=True)[:k])

top1 = _topk_weight(effective, 1)
top5 = _topk_weight(effective, 5)
top10 = _topk_weight(effective, 10)

# Single-name kill-shot on the largest effective position
top_sym, top_w = max(effective.items(), key=lambda kv: kv[1])
kill_shot_pct = top_w * 0.20  # 20% drop scenario on largest effective position

# --- render (shared toolkit) ------------------------------------------------
_rows = [
    (nb_pill("HHI", "accent"), f"{hhi_xray:.4f}"),
    (nb_pill("Effective-N", "accent"), f"{neff_xray:.2f}"),
    (nb_pill("Top-1 weight", "accent"), f"{top1 * 100:.2f}%"),
    (nb_pill("Top-5 weight", "accent"), f"{top5 * 100:.2f}%"),
    (nb_pill("Top-10 weight", "accent"), f"{top10 * 100:.2f}%"),
]
_ks_tone = "bad" if kill_shot_pct >= 0.03 else "warn" if kill_shot_pct >= 0.015 else "good"
nb_panel(
    "Concentration — top-K weights + single-name kill-shot",
    nb_table(["metric", "value"], _rows)
    + f"<div style='margin-top:9px'>{nb_pill(f'{top_sym} largest · {top_w * 100:.1f}%', 'accent')}"
    + nb_pill(f"−20% on {top_sym} → portfolio −{kill_shot_pct * 100:.2f}%", _ks_tone)
    + "<span style='opacity:.7;font:12px/1.5 ui-sans-serif,system-ui'>"
    + " — single-name kill-shot on the largest effective position.</span></div>",
    subtitle=("Computed in-notebook on the §3 effective positions — same math the "
              "risk.concentration router runs, minus the look-through re-call."),
    tone=_ks_tone,
    links=["hhi", "diversification"],
)


Concentration (post-look-through, computed in-notebook):
  HHI                         0.0598
  Effective-N                  16.72
  Top-1 weight                12.00%
  Top-5 weight                47.69%
  Top-10 weight               68.28%

Story-side single-name kill-shot:
  Largest effective position: MSFT at 12.0%
  If MSFT drops 20%: portfolio hit = 2.40%

(obb.portfolio_intel.risk.concentration router IS available and does
 the same math — but it re-calls xray.look_through internally, which
 stalls on the ETF-Holdings-exhausted path for me. Computing directly
 from the effective_positions we already have is faster and honest.)


## 8. Portfolio construction — turning the x-ray into weights

The x-ray told me *what I'm exposed to*. This section turns that into
*what I should hold* — a defendable weighting policy, not a black-box
optimizer.

Three moves, in order:

1. **Coverage guard first.** If too little of the book resolved to
   attributed effective positions, we **refuse to emit targets** — the
   same loud-empty discipline the risk guard uses in §9. Constructing
   weights on a thin look-through just launders the error into your
   allocation.
2. **Target bands, not points.** N-PORT is quarterly and ~30-60 days
   lagged, so a target of "MSFT 8.34%" is false precision. We emit a
   **band** (point ± 1.5pp) and rebalance only when a holding drifts
   *outside* its band — not on every N-PORT refresh.
3. **Single-name cap.** Any effective issuer above the cap (default 6%)
   is flagged **TRIM**, and the freed weight is reported for you to
   redistribute. This is the concentration limit §5-§7 argued for, made
   actionable.

> **📖 Rebalancing** — resetting holdings back toward target weights after drift. Band-based rebalancing (act only outside a tolerance) beats calendar rebalancing on both turnover and tax drag. [Investopedia →](https://www.investopedia.com/terms/r/rebalancing.asp)

**Why this is the *safe* use of N-PORT for construction:** the filing is
authoritative for *structure* (who's inside each ETF, how the sectors
split), so it's the right source for setting name/sector **targets and
caps**. It is *not* fresh enough to be a live rebalancing trigger, so we
never treat its lagged weights as knife-edge points — we widen them into
bands and cap the extremes. **Structure from N-PORT, execution from live
prices** (the §6 pipeline).

*The code cell below emits the target bands + caps, or refuses if
coverage is too thin.*


In [ ]:
# [Phase B / NB03 §8] Portfolio construction — target bands + concentration caps
# Turns the §3 N-PORT effective positions into an ACTIONABLE weighting policy:
#   - a coverage guard that REFUSES to emit targets if too much of the book
#     is un-attributed (same loud-empty discipline as the §9 risk guard),
#   - per-name TARGET BANDS (point ± tolerance) rather than knife-edge points,
#     because N-PORT is quarterly + ~30-60d lagged — precise-to-the-bp targets
#     would be false precision,
#   - a single-name concentration CAP that flags names to trim and reports the
#     freed weight to redistribute.
#
# Intentionally NOT an optimizer. It's a rules-based policy you can defend
# line-by-line — the honest use of look-through data for construction.

# --- policy knobs (edit these to your mandate) ------------------------------
MAX_SINGLE_NAME = 0.06   # hard cap on any one effective issuer (6%)
TOP5_CAP        = 0.35   # soft cap on the top-5 effective concentration
BAND_ABS        = 0.015  # ±1.5pp tolerance band absorbs N-PORT lag
MIN_COVERAGE    = 0.90   # refuse to emit targets if <90% of book is attributed

# --- coverage guard: don't construct on a thin look-through -----------------
_cov = sum(effective.values())                       # attributed fraction
_opaque_w = sum(p["weight"] for p in basket if p["symbol"] in (non_nport or []))

if _cov < MIN_COVERAGE:
    construction_targets = None
    nb_panel(
        "Portfolio construction — REFUSED (coverage too thin)",
        f"<div>Only <b>{_cov * 100:.1f}%</b> of the book resolved to attributed "
        f"effective positions (threshold {MIN_COVERAGE * 100:.0f}%). Emitting "
        "weight targets on this would bake in look-through error.</div>"
        f"<div style='margin-top:9px'>{nb_pill('NO TARGETS EMITTED', 'bad')}"
        "<span style='opacity:.75;font:12px/1.5 ui-sans-serif,system-ui'>"
        " — close the coverage gap (missing N-PORT filers / unresolved names) "
        "before trusting these for construction.</span></div>",
        subtitle="Same loud-empty discipline as the §9 risk guard — refuse, don't fabricate.",
        tone="bad",
        badge="REFUSED",
        links=["look_through", "diversification"],
    )
else:
    # --- build target bands + apply the single-name cap ---------------------
    _sorted = sorted(effective.items(), key=lambda kv: -kv[1])
    _freed = 0.0
    _over = 0
    construction_targets = {}
    for _name, _w in _sorted:
        if _w > MAX_SINGLE_NAME:
            _target = MAX_SINGLE_NAME
            _freed += _w - MAX_SINGLE_NAME
            _over += 1
            _status = "TRIM"
        else:
            _target = _w
            _status = "ok"
        construction_targets[_name] = {
            "current": _w,
            "target": _target,
            "band_lo": max(0.0, _target - BAND_ABS),
            "band_hi": _target + BAND_ABS,
            "status": _status,
        }

    _top5_now = sum(w for _, w in _sorted[:5])

    # --- render (shared toolkit) --------------------------------------------
    def _status_pill(s):
        return nb_pill("TRIM", "bad") if s == "TRIM" else nb_pill("ok", "good")

    _rows = []
    for _name, _w in _sorted[:15]:
        t = construction_targets[_name]
        band = f"{t['band_lo'] * 100:.1f}–{t['band_hi'] * 100:.1f}%"
        _rows.append((
            nb_pill(str(_name)[:34], "accent"),
            f"{t['current'] * 100:.2f}%",
            band,
            _status_pill(t["status"]),
        ))

    _cap_tone = "good" if _over == 0 else "warn"
    _top5_tone = "good" if _top5_now <= TOP5_CAP else "warn"
    _opaque_note = (
        f"{nb_pill(f'opaque {_opaque_w * 100:.1f}%', 'warn')}"
        "<span style='opacity:.7;font:12px/1.5 ui-sans-serif,system-ui'>"
        " — un-decomposed positions (e.g. GLD) sized as their own real holding; "
        "look-through caps don't apply to them.</span>"
        if _opaque_w > 0 else ""
    )
    nb_panel(
        "Portfolio construction — target bands + concentration caps",
        nb_table(["issuer / bucket", "current", "target band", "status"], _rows)
        + "<div style='margin-top:9px'>"
        + nb_pill(f"coverage {_cov * 100:.1f}%", "good")
        + nb_pill(f"names over {MAX_SINGLE_NAME * 100:.0f}% cap: {_over}", _cap_tone)
        + nb_pill(f"freed to redistribute {_freed * 100:.2f}%", _cap_tone)
        + nb_pill(f"top-5 {_top5_now * 100:.1f}% / {TOP5_CAP * 100:.0f}% cap", _top5_tone)
        + "</div>"
        + (f"<div style='margin-top:6px'>{_opaque_note}</div>" if _opaque_note else "")
        + "<div style='margin-top:6px;opacity:.6;font:11px/1.5 ui-sans-serif,system-ui'>"
        + f"Bands are ±{BAND_ABS * 100:.1f}pp — wide enough to absorb N-PORT's "
        + "quarterly / ~30-60d lag. Rebalance on band breach, not on every refresh. "
        + f"{unknown_count} tail constituents were Unknown/Other in §4 (each &lt;0.05%).</div>",
        subtitle=("Rules-based policy on the §3 effective positions — defendable "
                  "line-by-line, not a black-box optimizer."),
        tone=_cap_tone,
        badge=f"{_over} to trim" if _over else "within caps",
        links=["diversification", "hhi", "position"],
    )


issuer / bucket,current,target band,status
MSFT,12.00%,4.5–7.5%,TRIM
NVDA,10.00%,4.5–7.5%,TRIM
TAIL_VTI,9.32%,4.5–7.5%,TRIM
TAIL_BND,8.37%,4.5–7.5%,TRIM
GOOGL,8.00%,4.5–7.5%,TRIM
AAPL,8.00%,4.5–7.5%,TRIM
AMD,6.00%,4.5–7.5%,ok
GLD,3.00%,1.5–4.5%,ok
TAIL_QQQ,1.99%,0.5–3.5%,ok
BOND_BND,1.60%,0.1–3.1%,ok


## 9. Silent-failure guards — why the numbers can be trusted

`obb.portfolio_intel.risk` was hardened in PR [#905](https://github.com/prajoria/OpenBB/issues/905) against three ways
these metrics silently return garbage:

1. **Partial-book variance** — if only 7 of 10 positions have valid
   price history, the covariance matrix is undefined. Old code
   substituted a rank-deficient matrix; new code refuses and tells
   you which positions failed.
2. **NaN in covariance / returns / benchmark** — same-shape refusal.
3. **NaN / Inf prices in the input** — refused at the door.

*The code cell below deliberately corrupts one price series in the
basket and re-runs `.metrics` to show the refusal path fires (not a
silent-zero result).*

In [ ]:
# [Phase B / NB03 §9] Silent-failure guard — NaN in prices must refuse, not silent-zero
# From PR #905 hardening: portfolio_intel.risk.metrics refuses to
# silently compute against NaN-poisoned inputs. Demonstrate by
# poisoning one position's returns with NaN and asserting the guard fires.
# (Uses a local `_pr` so the clean §6 `r`/`res` used by §11 stay intact.)
import math
import copy
from openbb import obb
import warnings; warnings.filterwarnings("ignore")

def _esc(s):
    return str(s).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

# Clone the returns_source we built above; poison the first position
if returns_source and len(returns_source) > 1:
    poisoned = copy.deepcopy(returns_source)
    victim = list(poisoned.keys())[0]
    poisoned[victim] = [math.nan] * len(poisoned[victim])
    try:
        _pr = obb.portfolio_intel.risk.metrics(
            basket=basket_for_risk,
            returns_source=poisoned,
            benchmark_returns=benchmark_returns,
        )
        _vol = getattr(_pr.results, "annualized_volatility", None)
        if _vol is None or (isinstance(_vol, float) and math.isnan(_vol)):
            _guard_tone, _guard_badge = "good", "GUARD FIRED"
            _guard_msg = (f"vol reported as <code>{_vol}</code> (loud-empty) — "
                          "the metric refuses to fabricate a number.")
        else:
            _guard_tone, _guard_badge = "bad", "SILENT ZERO"
            _guard_msg = (f"UNEXPECTED: computation succeeded with vol=<code>{_vol}</code> "
                          "— silent-guard behavior regressed.")
    except Exception as exc:
        _guard_tone, _guard_badge = "good", "GUARD RAISED"
        _guard_msg = (f"raised <code>{type(exc).__name__}</code>: "
                      f"{_esc(str(exc)[:160])}")
    nb_panel(
        "Silent-failure guard — NaN prices must refuse, not silent-zero",
        f"<div>Poisoned <b>{_esc(victim)}</b>'s return series with all-NaN values, "
        f"then re-ran <code>risk.metrics</code>.</div>"
        f"<div style='margin-top:9px'>{nb_pill(_guard_badge, _guard_tone)}"
        f"<span style='opacity:.8;font:12px/1.5 ui-sans-serif,system-ui'> {_guard_msg}</span></div>",
        subtitle=("PR #905 hardening: partial-book / NaN-covariance / NaN-price inputs "
                  "are refused at the door."),
        tone=_guard_tone,
        badge=_guard_badge,
        links=[],
    )
else:
    nb_panel(
        "Silent-failure guard",
        "<i style='opacity:.6'>Skipped — need returns_source populated from §6.</i>",
        tone="warn",
        badge="SKIPPED",
    )


Poisoning MSFT's return series with all-NaN values...
  OK: guard fired — vol reported as None (loud-empty)


## 10. Reading the numbers

A quick reference — the two-sentence version of each metric, in Sam's
words:

- **HHI**: sum of squared weights. Higher means more concentrated. Ours
  roughly doubles when we look through the ETFs.
- **Effective N**: 1/HHI. "How many equal-weight positions am I really
  holding?"
- **Sharpe**: excess return over risk-free per unit of vol. Under
  ~0.5, your risk isn't being paid for.
- **Max drawdown**: worst historical peak-to-trough. If it hurts to
  read the number, it will hurt more to live through it.
- **Tracking error**: how far you wander from your benchmark. Only worth
  it if your Sharpe beats the benchmark's.

## 11. Save state for NB04, NB05, NB06

Everything downstream picks up:
- `.notebook_state/basket.json` — the 10 positions (already written)
- `.notebook_state/xray.pkl` — the flattened positions + sector weights
- `.notebook_state/risk.pkl` — the four risk metrics

*The code cell below pickles the x-ray + risk artifacts.*

In [ ]:
# [Phase B / NB03 §11] Save state for NB04/NB05/NB06
# Pickle safety note (same as NB02 §9): file lives under
# .notebook_state/ (gitignored), trusted-local-only, never shipped.
# Long-term: msgspec/pydantic schema would remove pickle entirely.
import pickle  # noqa: S403  # trusted local artifact; safety documented above
from pathlib import Path

state = Path(".notebook_state")
state.mkdir(exist_ok=True)

# xray artifact — dict form (simpler to consume in later notebooks)
xray_artifact = {
    "basket": basket,
    "effective_positions": effective,  # dict[str,float]  post-look-through
    "naive_by_sector": naive_by_sector,
    "xray_by_sector": xray_by_sector,
    "hhi_raw": hhi_raw,
    "hhi_xray": hhi_xray,
    "neff_raw": neff_raw,
    "neff_xray": neff_xray,
    "construction_targets": construction_targets,  # §8 band/cap policy (or None if coverage refused)
}
(state / "xray.pkl").write_bytes(pickle.dumps(xray_artifact))

# risk artifact — only saved if the risk cell succeeded
try:
    risk_artifact = {
        "basket_for_risk": basket_for_risk,
        "returns_source": returns_source,
        "benchmark_returns": benchmark_returns,
        # r.results from §6 — attribute-carrying object
        "metrics_dump": {
            name: getattr(r.results, name)
            for name in dir(r.results)
            if not name.startswith("_") and not callable(getattr(r.results, name, None))
        },
    }
    (state / "risk.pkl").write_bytes(pickle.dumps(risk_artifact))
    saved_risk = True
except NameError:
    saved_risk = False

# --- render (shared toolkit) ------------------------------------------------
_xray_sz = (state / "xray.pkl").stat().st_size
_rows = [(nb_pill("xray.pkl", "accent"), f"{_xray_sz:,} bytes",
          "flattened positions + sector weights + construction targets")]
if saved_risk:
    _risk_sz = (state / "risk.pkl").stat().st_size
    _rows.append((nb_pill("risk.pkl", "accent"), f"{_risk_sz:,} bytes",
                  "risk metrics dump"))
    _tone, _badge = "good", "SAVED"
else:
    _rows.append((nb_pill("risk.pkl", "warn"), "—", "not written — §6 didn't complete"))
    _tone, _badge = "warn", "PARTIAL"
nb_panel(
    "State saved for NB04 / NB05 / NB06",
    nb_table(["artifact", "size", "contents"], _rows)
    + "<div style='margin-top:6px;opacity:.6;font:11px/1.5 ui-sans-serif,system-ui'>"
    + f"Written under <code>{state}/</code> (gitignored, trusted-local-only). "
    + "basket.json was written in §1.</div>",
    subtitle="Downstream notebooks pick these up in their §0 load cells.",
    tone=_tone,
    badge=_badge,
    links=[],
)


artifact,size,contents
xray.pkl,"14,605 bytes",flattened positions + sector weights + construction targets
risk.pkl,"15,828 bytes",risk metrics dump


---

## What is NOT in this notebook

- **Cash positions.** Sam holds equity + ETFs only in this basket; cash-as-a-position support is [#903](https://github.com/prajoria/OpenBB/issues/903) and not shipped.
- **Short positions.** Same story — [#904](https://github.com/prajoria/OpenBB/issues/904).
- **Factor decomposition (Fama-French, Carhart).** The `openbb_famafrench` extension exists but isn't wired into portfolio_intel risk yet.

## Preview of NB04

Now I know what I own and how concentrated I am. The picture is worse than I thought. But the picture is static — it's a snapshot of my exposures. Two things move that snapshot every week: **events** on the calendar (earnings, dividends, splits) and **smart-money activity** (13F changes, insider transactions, government trades). In NB04 we overlay both.


## 📚 Further reading

Every Investopedia link cited in this notebook, plus canonical references:

- [Portfolio — Investopedia](https://www.investopedia.com/terms/p/portfolio.asp)
- [Position — Investopedia](https://www.investopedia.com/terms/p/position.asp)
- [Weighted average — Investopedia](https://www.investopedia.com/terms/w/weighted.asp)
- [Diversification — Investopedia](https://www.investopedia.com/terms/d/diversification.asp)
- [Sector breakdown — Investopedia](https://www.investopedia.com/terms/s/sectorbreakdown.asp)
- [Look-through — Investopedia](https://www.investopedia.com/terms/l/look-through-earnings.asp)
- [Herfindahl-Hirschman Index — Investopedia](https://www.investopedia.com/terms/h/hhi.asp)
- [Sharpe ratio — Investopedia](https://www.investopedia.com/terms/s/sharperatio.asp)
- [Volatility — Investopedia](https://www.investopedia.com/terms/v/volatility.asp)
- [Maximum drawdown — Investopedia](https://www.investopedia.com/terms/m/maximum-drawdown-mdd.asp)
- [Tracking error — Investopedia](https://www.investopedia.com/terms/t/trackingerror.asp)
- [Benchmark — Investopedia](https://www.investopedia.com/terms/b/benchmark.asp)
- [Risk-free rate — Investopedia](https://www.investopedia.com/terms/r/risk-freerate.asp)
- [Correlation — Investopedia](https://www.investopedia.com/terms/c/correlation.asp)

**Canonical references beyond Investopedia:**

- Grinold, R. C. & Kahn, R. N. — *Active Portfolio Management*, 2nd ed.,
  ch. 3 ("Expected Returns and the Arithmetic of Active Management"). The
  reference text for the effective-breadth / effective-N intuition and the
  Information Ratio framing that generalizes Sharpe for active books.
- Markowitz, H. — "Portfolio Selection," *Journal of Finance* 7(1), 1952.
  The original mean-variance paper; every risk number in this notebook
  descends from it.
